# StorageLLM Gemma MXFP4-MOE Colab Runtime Benchmark

Downloads the latest engine source, downloads `storagejuju/gemma-4-26b-a4b-it-mxfp4-moe-juju`, builds the native server/probe, auto-selects CUDA on GPU runtimes unless overridden, measures attach/load time, bounded generation time, PPL/smoke time, CPU/RAM samples, and trace bottlenecks. Outputs stay under `/content/STORAGELLM_COLAB_BENCH`.


In [ ]:
# Cell 1 - config, dependencies, latest engine checkout, model download
import os, sys, json, time, shlex, shutil, pathlib, subprocess, re
from datetime import datetime

ENGINE_REPO_URL = os.environ.get('ENGINE_REPO_URL', 'https://github.com/jujumelona/storage.llm.git')
ENGINE_GIT_REF = os.environ.get('ENGINE_GIT_REF', 'main')
MODEL_REPO_ID = os.environ.get('MODEL_REPO_ID', 'storagejuju/gemma-4-26b-a4b-it-mxfp4-moe-juju')
MODEL_REVISION = os.environ.get('MODEL_REVISION', 'main')
RUN_ROOT = pathlib.Path(os.environ.get('RUN_ROOT', '/content/STORAGELLM_COLAB_BENCH'))
ENGINE_DIR = RUN_ROOT / 'storage_llm'
MODEL_DIR = RUN_ROOT / 'models' / MODEL_REPO_ID.replace('/', '__')
LOG_DIR = RUN_ROOT / 'logs' / datetime.utcnow().strftime('%Y%m%d_%H%M%S')
PORT = int(os.environ.get('STORAGELLM_PORT', '18080'))
BACKEND = os.environ.get('STORAGELLM_BACKEND', 'auto').strip().lower()
PLATFORM = os.environ.get('STORAGELLM_PLATFORM', 'linux').strip().lower()
TRACE = os.environ.get('STORAGELLM_TRACE', '1').lower() not in ('0', 'false', 'no')
USE_MARCH_NATIVE = os.environ.get('STORAGELLM_MARCH_NATIVE', '1').lower() not in ('0', 'false', 'no')
GEN_INPUT_IDS = [int(x) for x in os.environ.get('GEN_INPUT_IDS', '2').replace(',', ' ').split()]
GEN_MAX_TOKENS = int(os.environ.get('GEN_MAX_TOKENS', '1'))
GEN_TIMEOUT_S = float(os.environ.get('GEN_TIMEOUT_S', '60'))
PPL_INPUT_IDS = [int(x) for x in os.environ.get('PPL_INPUT_IDS', '').replace(',', ' ').split() if x.strip()]
PPL_TEXT = os.environ.get('PPL_TEXT', '')
PPL_MODE = os.environ.get('PPL_MODE', 'wikitext2').strip().lower()
PPL_DATASET = os.environ.get('PPL_DATASET', 'wikitext')
PPL_DATASET_FALLBACK = os.environ.get('PPL_DATASET_FALLBACK', 'Salesforce/wikitext')
PPL_DATASET_CONFIG = os.environ.get('PPL_DATASET_CONFIG', 'wikitext-2-raw-v1')
PPL_DATASET_SPLIT = os.environ.get('PPL_DATASET_SPLIT', 'test')
PPL_TEXT_CHARS = int(os.environ.get('PPL_TEXT_CHARS', '20000'))
PPL_MAX_TOKENS = int(os.environ.get('PPL_MAX_TOKENS', '32'))
PPL_TIMEOUT_S = float(os.environ.get('PPL_TIMEOUT_S', '900'))
SHOW_FULL_LOGS = os.environ.get('SHOW_FULL_LOGS', '1').lower() not in ('0', 'false', 'no')
KILL_SERVER_ON_TIMEOUT = os.environ.get('KILL_SERVER_ON_TIMEOUT', '1').lower() not in ('0', 'false', 'no')

RUN_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print('RUN_ROOT =', RUN_ROOT)
print('ENGINE_REPO_URL =', ENGINE_REPO_URL)
print('ENGINE_GIT_REF =', ENGINE_GIT_REF)
print('MODEL_REPO_ID =', MODEL_REPO_ID)
print('MODEL_DIR =', MODEL_DIR)
print('LOG_DIR =', LOG_DIR)


def run(cmd, cwd=None, env=None, check=True, log_name=None, timeout=None):
    if isinstance(cmd, (list, tuple)):
        printable = ' '.join(shlex.quote(str(x)) for x in cmd)
        popen_cmd = list(map(str, cmd))
        shell = False
    else:
        printable = cmd
        popen_cmd = cmd
        shell = True
    print('\n$ ' + printable)
    log_f = None
    log_path = None
    if log_name:
        log_path = LOG_DIR / log_name
        log_f = log_path.open('w', encoding='utf-8', errors='replace')
    t0 = time.perf_counter()
    proc = subprocess.Popen(
        popen_cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        shell=shell,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        errors='replace')
    lines = []
    try:
        for line in proc.stdout:
            print(line, end='')
            lines.append(line)
            if log_f:
                log_f.write(line)
        rc = proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        rc = proc.wait()
        lines.append(f'\nTIMEOUT after {timeout}s\n')
    finally:
        if log_f:
            log_f.close()
    elapsed = time.perf_counter() - t0
    print(f'[exit={rc} elapsed={elapsed:.3f}s]')
    if check and rc != 0:
        raise RuntimeError(f'command failed rc={rc}: {printable}')
    return {'rc': rc, 'elapsed_s': elapsed, 'output': ''.join(lines), 'log_path': str(log_path) if log_path else None}

run('apt-get update -y && apt-get install -y git g++ make curl procps zip', log_name='apt_install.log')
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub', 'hf_transfer', 'transformers', 'datasets', 'sentencepiece'], log_name='pip_install.log')
run('uname -a && lscpu && free -h && df -h /content && (nvidia-smi || true)', log_name='system_info.log', check=False)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

if ENGINE_DIR.exists() and (ENGINE_DIR / '.git').exists():
    run(['git', 'fetch', '--all', '--tags', '--prune'], cwd=ENGINE_DIR, log_name='git_fetch.log', check=False)
else:
    if ENGINE_DIR.exists():
        shutil.rmtree(ENGINE_DIR)
    run(['git', 'clone', ENGINE_REPO_URL, str(ENGINE_DIR)], log_name='git_clone.log')
run(['git', 'checkout', ENGINE_GIT_REF], cwd=ENGINE_DIR, log_name='git_checkout.log')
run(['git', 'pull', '--ff-only'], cwd=ENGINE_DIR, log_name='git_pull.log', check=False)

# Hotfix stale main revisions that still reference removed moe_io_config_t.device_memory_headroom_pct.
prefetch_plan_path = ENGINE_DIR / 'moe_engine/src/parts/prefetch_plan.cpp.inc'
if prefetch_plan_path.exists():
    prefetch_plan_text = prefetch_plan_path.read_text(encoding='utf-8')
    stale_headroom = 'engine->io_config.device_memory_headroom_pct'
    if stale_headroom in prefetch_plan_text:
        replacement = """uint64_t headroom_pct = 10u;
        if (engine->backend_caps.backend == moe_BACKEND_VULKAN && engine->io_config.vulkan_memory_headroom_pct > 0) {
            headroom_pct = engine->io_config.vulkan_memory_headroom_pct;
        } else if (engine->backend_caps.backend == moe_BACKEND_LEVEL_ZERO && engine->io_config.level_zero_memory_headroom_pct > 0) {
            headroom_pct = engine->io_config.level_zero_memory_headroom_pct;
        }"""
        prefetch_plan_text, patched_count = re.subn(
            r"const uint64_t headroom_pct = engine->io_config\.device_memory_headroom_pct > 0\s*\n\s*\? engine->io_config\.device_memory_headroom_pct : 10u;",
            replacement,
            prefetch_plan_text,
        )
        if patched_count <= 0:
            raise RuntimeError('stale device_memory_headroom_pct found but patch pattern did not match')
        prefetch_plan_path.write_text(prefetch_plan_text, encoding='utf-8')
        print('patched stale prefetch_plan headroom field:', prefetch_plan_path)
head = run(['git', 'rev-parse', 'HEAD'], cwd=ENGINE_DIR, log_name='git_head.log')['output'].strip().splitlines()[-1]
print('ENGINE_HEAD =', head)

from huggingface_hub import snapshot_download
MODEL_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.perf_counter()
local = snapshot_download(
    repo_id=MODEL_REPO_ID,
    repo_type='model',
    revision=MODEL_REVISION,
    local_dir=str(MODEL_DIR),
    ignore_patterns=['runtime_asset_upload_stage/*', '.git/*'],
    resume_download=True)
download_s = time.perf_counter() - t0
print('MODEL_LOCAL =', local)
print(f'MODEL_DOWNLOAD_OR_ATTACH_S = {download_s:.3f}')
required = [
    'gemma-4-26B-A4B-it-MXFP4_MOE.juju',
    'gemma-4-26B-A4B-it-MXFP4_MOE.juju.idx',
    'config.json', 'tokenizer.json', 'tokenizer_config.json', 'generation_config.json', 'chat_template.jinja'
]
model_files = []
for p in sorted(MODEL_DIR.rglob('*')):
    if p.is_file():
        model_files.append({'path': str(p.relative_to(MODEL_DIR)), 'bytes': p.stat().st_size})
missing = [x for x in required if not (MODEL_DIR / x).exists()]
summary = {'engine_head': head, 'model_repo': MODEL_REPO_ID, 'model_dir': str(MODEL_DIR), 'download_s': download_s, 'missing': missing, 'files': model_files}
(LOG_DIR / 'download_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps({'missing_required': missing, 'file_count': len(model_files), 'bytes_total': sum(x['bytes'] for x in model_files)}, indent=2))
if missing:
    raise RuntimeError('missing model files: ' + ', '.join(missing))



In [ ]:
# Cell 2 - build native probe/server from the checked-out latest engine source
import os, pathlib
BUILD_DIR = ENGINE_DIR / 'build' / 'linux'
BUILD_DIR.mkdir(parents=True, exist_ok=True)
PROBE_EXE = BUILD_DIR / 'moe_pc_engine_probe'
SERVER_EXE = BUILD_DIR / 'moe_pc_engine_server'

COMMON_SOURCES = [
    'moe_engine/src/moe_pc_engine.cpp',
    'moe_engine/src/hardcode_constants.cpp',
    'moe_engine/src/hardcode_layers.cpp',
    'moe_engine/src/hardcode_parts.cpp',
    'moe_engine/src/hardcode_projection_shapes.cpp',
    'moe_engine/src/hardcode_raw_spans.cpp',
    'moe_engine/src/hardcode_summary.cpp',
    'moe_engine/src/model_shape.cpp',
    'moe_engine/native/scale4.cpp',
    'moe_engine/src/tools/storage_f8.cpp',
    'engine_core/core/mmap_loader.cpp',
    'engine_core/kv/kv_qkv.cpp',
    'engine_core/kv/qkv_attention.cpp',
    'engine_core/kv/qkv_codebook.cpp',
    'engine_core/kv/qkv_dequantize.cpp',
    'engine_core/kv/qkv_helpers.cpp',
    'engine_core/kv/qkv_matrix.cpp',
    'engine_core/kv/qkv_packing.cpp',
    'engine_core/kv/qkv_quantize.cpp',
    'engine_core/kv/qkv_state.cpp',
    'engine_core/kv/qkv_thread_pool.cpp',
    'loader/file_read.cpp',
    'loader/json_find.cpp',
    'loader/json_match.cpp',
    'loader/json_value.cpp',
    'loader/manifest_key.cpp',
    'loader/manifest_lookup.cpp',
    'loader/manifest_projection.cpp',
    'loader/path_join.cpp',
]
INCLUDES = ['-I.', '-Iloader', '-Iengine_core/core', '-Iengine_core', '-Imoe_engine/include', '-Imoe_engine/native']
CXXFLAGS = ['-std=c++17', '-O3', '-DNDEBUG', '-pthread']
if USE_MARCH_NATIVE:
    CXXFLAGS.append('-march=native')
extra = os.environ.get('STORAGELLM_EXTRA_CXXFLAGS', '').split()
CXXFLAGS.extend(extra)

def build_one(entry, out, log_name):
    cmd = ['g++'] + CXXFLAGS + INCLUDES + [entry] + COMMON_SOURCES + ['-o', str(out)]
    return run(cmd, cwd=ENGINE_DIR, log_name=log_name)

build_probe = build_one('moe_engine/examples/pc_engine_probe.cpp', PROBE_EXE, 'build_probe.log')
build_server = build_one('moe_engine/examples/pc_engine_server.cpp', SERVER_EXE, 'build_server.log')
print('PROBE_EXE =', PROBE_EXE, PROBE_EXE.stat().st_size)
print('SERVER_EXE =', SERVER_EXE, SERVER_EXE.stat().st_size)


In [ ]:
# Cell 3 - probe model root, start server, measure attach/load readiness
import json, urllib.request, urllib.error, socket, subprocess, os, signal, time

PROBE_LOG = LOG_DIR / 'probe.log'
probe_t0 = time.perf_counter()
probe = run([str(PROBE_EXE), '--backend', BACKEND, '--platform', PLATFORM, '--model-root', str(MODEL_DIR)], cwd=ENGINE_DIR, log_name='probe.log', check=False)
probe_s = time.perf_counter() - probe_t0
print('PROBE_RC =', probe['rc'], 'PROBE_S =', probe_s)
if probe['rc'] != 0:
    raise RuntimeError('probe failed; inspect ' + str(PROBE_LOG))

SERVER_LOG = LOG_DIR / 'server.log'
PID_FILE = LOG_DIR / 'server.pid'

def kill_pid(pid):
    try:
        os.kill(int(pid), signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass

if PID_FILE.exists():
    old = PID_FILE.read_text().strip()
    if old:
        kill_pid(old)

env = os.environ.copy()
if TRACE:
    env['STORAGELLM_TRACE'] = '1'
else:
    env.pop('STORAGELLM_TRACE', None)
server_cmd = [str(SERVER_EXE), '--host', '127.0.0.1', '--port', str(PORT), '--backend', BACKEND, '--platform', PLATFORM, '--model-id', MODEL_REPO_ID.split('/')[-1], '--model-root', str(MODEL_DIR)]
print('$ ' + ' '.join(shlex.quote(x) for x in server_cmd))
server_log_f = SERVER_LOG.open('w', encoding='utf-8', errors='replace')
server_proc = subprocess.Popen(server_cmd, cwd=str(ENGINE_DIR), env=env, stdout=server_log_f, stderr=subprocess.STDOUT, text=True)
PID_FILE.write_text(str(server_proc.pid), encoding='utf-8')
print('SERVER_PID =', server_proc.pid)

def health(timeout=1.0):
    raw = urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=timeout).read().decode()
    return json.loads(raw)

ready_samples = []
load_t0 = time.perf_counter()
ready = None
for i in range(180):
    item = {'t': time.perf_counter() - load_t0}
    try:
        h = health(timeout=1.0)
        item.update({k: h.get(k) for k in ['modelLoading', 'modelReady', 'generationReady', 'chatGraphReady', 'modelLoadStage', 'processRssBytes', 'ramUsedBytes', 'storageTierBytes', 'generationActive']})
        ready_samples.append(item)
        if h.get('modelReady'):
            ready = h
            break
        if h.get('modelFailed'):
            ready = h
            break
    except Exception as e:
        item['error'] = type(e).__name__ + ':' + str(e)
        ready_samples.append(item)
    time.sleep(1)
load_s = time.perf_counter() - load_t0
attach_summary = {'server_pid': server_proc.pid, 'probe_s': probe_s, 'load_ready_s': load_s, 'ready': bool(ready and ready.get('modelReady')), 'health': ready, 'samples': ready_samples}
(LOG_DIR / 'attach_load_summary.json').write_text(json.dumps(attach_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'ready': attach_summary['ready'], 'probe_s': probe_s, 'load_ready_s': load_s, 'pid': server_proc.pid, 'health': {k: ready.get(k) for k in ['modelReady','generationReady','chatGraphReady','forwardAdapter','forwardAdapterExecutable','modelRootValid','modelRootPresentParts','modelRootExpectedParts','ramUsedBytes','storageTierBytes'] if ready}}, ensure_ascii=False, indent=2))
if not attach_summary['ready']:
    raise RuntimeError('server did not become ready; inspect ' + str(SERVER_LOG))


In [ ]:
# Cell 4 - bounded 1-token generation with CPU/RAM/phase sampling
import threading, re

def ps_line(pid):
    try:
        return subprocess.check_output(['ps', '-p', str(pid), '-o', 'pid=,stat=,pcpu=,pmem=,rss=,nlwp=,wchan='], text=True, timeout=1).strip()
    except Exception as e:
        return type(e).__name__ + ':' + str(e)

def sample_health(pid, start_time, out_jsonl, stop_flag, interval=1.0):
    samples = []
    while not stop_flag['stop']:
        item = {'t': time.perf_counter() - start_time, 'ps': ps_line(pid)}
        try:
            h = health(timeout=0.8)
            for k in ['generationActive','generationToken','generationLayer','generationPhase','generationStarted','generationCompleted','generationFailed','processRssBytes','availableRamBytes','ramUsedBytes','storageTierBytes','diskQueueDepth','pinnedQueueDepth','gpuQueueDepth','activeWorkers','stagingSlotDeficit']:
                item[k] = h.get(k)
        except Exception as e:
            item['health_error'] = type(e).__name__ + ':' + str(e)
        samples.append(item)
        with open(out_jsonl, 'a', encoding='utf-8') as f:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
        time.sleep(interval)
    return samples

def parse_cpu_from_ps(ps):
    try:
        parts = ps.split()
        return float(parts[2])
    except Exception:
        return None

def parse_rss_kb_from_ps(ps):
    try:
        parts = ps.split()
        return int(parts[4])
    except Exception:
        return None

pid = int(PID_FILE.read_text().strip())
gen_samples_path = LOG_DIR / 'generation_samples.jsonl'
if gen_samples_path.exists():
    gen_samples_path.unlink()
body = json.dumps({'input_ids': GEN_INPUT_IDS, 'max_output_tokens': GEN_MAX_TOKENS}).encode()
req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/responses', data=body, headers={'Content-Type': 'application/json'}, method='POST')
stop_flag = {'stop': False}
request_t0 = time.perf_counter()
samples_box = {'samples': []}

def poller():
    samples_box['samples'] = sample_health(pid, request_t0, gen_samples_path, stop_flag, 1.0)

thread = threading.Thread(target=poller, daemon=True)
thread.start()
gen_result = {'pid': pid, 'input_ids': GEN_INPUT_IDS, 'max_output_tokens': GEN_MAX_TOKENS}
try:
    raw = urllib.request.urlopen(req, timeout=GEN_TIMEOUT_S).read().decode()
    gen_result['ok'] = True
    gen_result['request_s'] = time.perf_counter() - request_t0
    gen_result['response'] = json.loads(raw)
except Exception as e:
    gen_result['ok'] = False
    gen_result['request_s'] = time.perf_counter() - request_t0
    gen_result['error'] = type(e).__name__ + ':' + str(e)
    if KILL_SERVER_ON_TIMEOUT:
        kill_pid(pid)
finally:
    stop_flag['stop'] = True
    thread.join(timeout=2)
try:
    gen_result['final_health'] = health(timeout=1.0)
except Exception as e:
    gen_result['final_health_error'] = type(e).__name__ + ':' + str(e)

samples = samples_box['samples']
cpus = [x for x in (parse_cpu_from_ps(s.get('ps','')) for s in samples) if x is not None]
rsses = [x for x in (parse_rss_kb_from_ps(s.get('ps','')) for s in samples) if x is not None]
gen_result['sample_count'] = len(samples)
gen_result['max_ps_cpu_pct'] = max(cpus) if cpus else None
gen_result['max_ps_rss_mb'] = max(rsses) / 1024.0 if rsses else None
gen_result['phases'] = sorted(set(str(s.get('generationPhase')) for s in samples if s.get('generationPhase') is not None))
gen_result['samples_tail'] = samples[-12:]
(LOG_DIR / 'generation_result.json').write_text(json.dumps(gen_result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({k: gen_result.get(k) for k in ['ok','request_s','max_ps_cpu_pct','max_ps_rss_mb','phases','response','error','samples_tail']}, ensure_ascii=False, indent=2))
if not gen_result['ok']:
    raise RuntimeError('generation did not finish inside timeout; logs saved under ' + str(LOG_DIR))


In [ ]:
# Cell 5 - bounded WikiText-2 PPL measurement with attach/time/resource samples
import re, urllib.request, threading


def latest_generated_token_from_log():
    if not SERVER_LOG.exists():
        return None
    text = SERVER_LOG.read_text(encoding='utf-8', errors='replace')
    matches = re.findall(r'decode_sample[^\n]*next_token=(\d+)', text)
    return int(matches[-1]) if matches else None


def build_wikitext_ppl_body():
    from datasets import load_dataset
    from transformers import AutoTokenizer

    dataset_errors = []
    ds = None
    dataset_used = None
    for dataset_name in [PPL_DATASET, PPL_DATASET_FALLBACK]:
        if not dataset_name:
            continue
        try:
            ds = load_dataset(dataset_name, PPL_DATASET_CONFIG, split=PPL_DATASET_SPLIT)
            dataset_used = dataset_name
            break
        except Exception as exc:
            dataset_errors.append(f'{dataset_name}:{type(exc).__name__}:{exc}')
    if ds is None:
        raise RuntimeError('failed to load WikiText dataset: ' + ' | '.join(dataset_errors))

    texts = []
    for row in ds:
        text = row.get('text', '') if isinstance(row, dict) else ''
        if text and text.strip():
            texts.append(text)
        if sum(len(x) + 2 for x in texts) >= PPL_TEXT_CHARS:
            break
    eval_text = '\n\n'.join(texts)[:PPL_TEXT_CHARS]
    if not eval_text.strip():
        raise RuntimeError('WikiText dataset loaded but no non-empty text rows were found')

    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True, use_fast=True)
    input_ids = tokenizer(eval_text, add_special_tokens=False).input_ids[:PPL_MAX_TOKENS]
    if len(input_ids) < 2:
        raise RuntimeError(f'PPL input is too short after tokenization: {len(input_ids)} tokens')
    return {'input_ids': input_ids}, {
        'mode': 'wikitext2',
        'dataset': dataset_used,
        'dataset_config': PPL_DATASET_CONFIG,
        'split': PPL_DATASET_SPLIT,
        'tokenizer_dir': str(MODEL_DIR),
        'token_count': len(input_ids),
        'char_count': len(eval_text),
        'max_tokens': PPL_MAX_TOKENS,
        'text_chars_limit': PPL_TEXT_CHARS,
    }


def build_trace_pair_ppl_body(reason=None):
    next_token = latest_generated_token_from_log()
    ids = []
    if GEN_INPUT_IDS:
        ids.append(int(GEN_INPUT_IDS[-1]))
    if next_token is not None:
        ids.append(int(next_token))
    while len(ids) < 2:
        ids.append(ids[-1] if ids else 2)
    return {'input_ids': ids[:2]}, {
        'mode': 'trace_pair_sanity' if reason is None else 'trace_pair_sanity_fallback',
        'token_count': 2,
        'reason': reason,
        'next_token_from_log': next_token,
    }


if PPL_TEXT:
    ppl_body = {'input': PPL_TEXT}
    ppl_source = {'mode': 'explicit_text', 'char_count': len(PPL_TEXT)}
elif PPL_INPUT_IDS:
    ppl_body = {'input_ids': PPL_INPUT_IDS}
    ppl_source = {'mode': 'explicit_input_ids', 'token_count': len(PPL_INPUT_IDS)}
elif PPL_MODE in ('wikitext', 'wikitext2', 'wikitext-2', 'wiki2'):
    try:
        ppl_body, ppl_source = build_wikitext_ppl_body()
    except Exception as exc:
        ppl_body, ppl_source = build_trace_pair_ppl_body('wikitext_setup_failed:' + type(exc).__name__ + ':' + str(exc)[:500])
elif PPL_MODE in ('trace_pair', 'trace-pair', 'sanity'):
    ppl_body, ppl_source = build_trace_pair_ppl_body()
else:
    raise RuntimeError(f'unknown PPL_MODE={PPL_MODE!r}; use wikitext2, trace_pair, PPL_INPUT_IDS, or PPL_TEXT')

body_token_count = len(ppl_body.get('input_ids', [])) if isinstance(ppl_body.get('input_ids'), list) else None
print('PPL_SOURCE =', json.dumps(ppl_source, ensure_ascii=False))
print('PPL_BODY_SUMMARY =', json.dumps({
    'has_input_text': 'input' in ppl_body,
    'token_count': body_token_count,
    'first_ids': ppl_body.get('input_ids', [])[:8] if body_token_count else None,
    'last_ids': ppl_body.get('input_ids', [])[-8:] if body_token_count else None,
}, ensure_ascii=False))

pid = int(PID_FILE.read_text().strip())
ppl_samples_path = LOG_DIR / 'ppl_samples.jsonl'
if ppl_samples_path.exists():
    ppl_samples_path.unlink()
body = json.dumps(ppl_body).encode()
req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/perplexity', data=body, headers={'Content-Type': 'application/json'}, method='POST')
stop_flag = {'stop': False}
request_t0 = time.perf_counter()
samples_box = {'samples': []}
thread = threading.Thread(target=lambda: samples_box.update(samples=sample_health(pid, request_t0, ppl_samples_path, stop_flag, 1.0)), daemon=True)
thread.start()
ppl_result = {'pid': pid, 'body': ppl_body, 'source': ppl_source}
try:
    raw = urllib.request.urlopen(req, timeout=PPL_TIMEOUT_S).read().decode()
    ppl_result['ok'] = True
    ppl_result['request_s'] = time.perf_counter() - request_t0
    ppl_result['response'] = json.loads(raw)
except Exception as e:
    ppl_result['ok'] = False
    ppl_result['request_s'] = time.perf_counter() - request_t0
    ppl_result['error'] = type(e).__name__ + ':' + str(e)
    if KILL_SERVER_ON_TIMEOUT:
        kill_pid(pid)
finally:
    stop_flag['stop'] = True
    thread.join(timeout=2)
try:
    ppl_result['final_health'] = health(timeout=1.0)
except Exception as e:
    ppl_result['final_health_error'] = type(e).__name__ + ':' + str(e)

samples = samples_box['samples']
cpus = [x for x in (parse_cpu_from_ps(s.get('ps','')) for s in samples) if x is not None]
rsses = [x for x in (parse_rss_kb_from_ps(s.get('ps','')) for s in samples) if x is not None]
ppl_result['sample_count'] = len(samples)
ppl_result['max_ps_cpu_pct'] = max(cpus) if cpus else None
ppl_result['max_ps_rss_mb'] = max(rsses) / 1024.0 if rsses else None
ppl_result['phases'] = sorted(set(str(s.get('generationPhase')) for s in samples if s.get('generationPhase') is not None))
ppl_result['samples_tail'] = samples[-12:]
(LOG_DIR / 'ppl_result.json').write_text(json.dumps(ppl_result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({k: ppl_result.get(k) for k in ['ok','request_s','source','max_ps_cpu_pct','max_ps_rss_mb','phases','response','error','samples_tail']}, ensure_ascii=False, indent=2))
if not ppl_result['ok']:
    raise RuntimeError('PPL did not finish inside timeout; logs saved under ' + str(LOG_DIR))


In [ ]:
# Cell 6 - bottleneck parser, full log display/export
import collections, zipfile, math

def read_json(name):
    p = LOG_DIR / name
    return json.loads(p.read_text(encoding='utf-8')) if p.exists() else None

def extract_ms(line):
    keys = ['total_ms', 'generation_ms', 'lm_head_ms', 'gate_up_ms', 'down_ms', 'attention_ms', 'mlp_ms', 'ms']
    found = []
    for key in keys:
        for m in re.finditer(r'\b' + re.escape(key) + r'=([0-9]+(?:\.[0-9]+)?)', line):
            found.append((key, float(m.group(1))))
    if not found:
        return None
    priority = {'total_ms': 0, 'generation_ms': 1, 'lm_head_ms': 2, 'gate_up_ms': 3, 'down_ms': 4, 'attention_ms': 5, 'mlp_ms': 6, 'ms': 7}
    found.sort(key=lambda x: priority.get(x[0], 99))
    return found[0]

def parse_trace(path):
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        m = re.search(r'\[storagellm trace\]\s+(\S+)\s+(.*)', line)
        if not m:
            continue
        got = extract_ms(line)
        if got is None:
            continue
        key, value = got
        rows.append({'tag': m.group(1), 'metric': key, 'ms': value, 'line': line})
    return rows

gen = read_json('generation_result.json') or {}
ppl = read_json('ppl_result.json') or {}
attach = read_json('attach_load_summary.json') or {}
rows = parse_trace(SERVER_LOG)
by_tag = collections.defaultdict(lambda: {'sum_ms': 0.0, 'max_ms': 0.0, 'count': 0})
for r in rows:
    by_tag[r['tag']]['sum_ms'] += r['ms']
    by_tag[r['tag']]['max_ms'] = max(by_tag[r['tag']]['max_ms'], r['ms'])
    by_tag[r['tag']]['count'] += 1
summary_rows = sorted(((k, v['sum_ms'], v['max_ms'], v['count']) for k, v in by_tag.items()), key=lambda x: x[1], reverse=True)
top_events = sorted(rows, key=lambda x: x['ms'], reverse=True)[:40]
report = {
    'engine_head': head,
    'model_repo': MODEL_REPO_ID,
    'model_dir': str(MODEL_DIR),
    'probe_s': attach.get('probe_s'),
    'load_ready_s': attach.get('load_ready_s'),
    'generation_request_s': gen.get('request_s'),
    'generation_cpu_max_pct': gen.get('max_ps_cpu_pct'),
    'generation_rss_max_mb': gen.get('max_ps_rss_mb'),
    'generation_response': gen.get('response'),
    'ppl_request_s': ppl.get('request_s'),
    'ppl_cpu_max_pct': ppl.get('max_ps_cpu_pct'),
    'ppl_rss_max_mb': ppl.get('max_ps_rss_mb'),
    'ppl_response': ppl.get('response'),
    'top_tags': [{'tag': k, 'sum_ms': s, 'max_ms': m, 'count': c} for k, s, m, c in summary_rows[:40]],
    'top_events': top_events,
}
(LOG_DIR / 'bottleneck_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('=== SUMMARY ===')
print(json.dumps({k: report[k] for k in ['engine_head','model_repo','probe_s','load_ready_s','generation_request_s','generation_cpu_max_pct','generation_rss_max_mb','ppl_request_s','ppl_cpu_max_pct','ppl_rss_max_mb','ppl_response']}, ensure_ascii=False, indent=2))
print('\n=== TOP TAGS BY SUM_MS ===')
for item in report['top_tags'][:25]:
    print(f"{item['sum_ms']:10.3f} ms sum | {item['max_ms']:10.3f} ms max | {item['count']:5d} | {item['tag']}")
print('\n=== TOP EVENTS ===')
for item in top_events[:25]:
    print(f"{item['ms']:10.3f} ms | {item['tag']} | {item['line'][:240]}")

zip_base = RUN_ROOT / ('storagellm_colab_logs_' + LOG_DIR.name)
archive = shutil.make_archive(str(zip_base), 'zip', LOG_DIR)
print('\nLOG_ARCHIVE =', archive)
print('LOG_DIR =', LOG_DIR)

def show(path, max_bytes=5_000_000):
    path = pathlib.Path(path)
    if not path.exists():
        return
    size = path.stat().st_size
    print(f"\n===== {path.name} ({size} bytes) =====")
    text = path.read_text(encoding='utf-8', errors='replace')
    if SHOW_FULL_LOGS or size <= max_bytes:
        print(text)
    else:
        print(text[-max_bytes:])

for name in ['download_summary.json','probe.log','server.log','generation_result.json','ppl_result.json','bottleneck_report.json']:
    show(LOG_DIR / name)


In [ ]:
# Cell 7 - optional cleanup; keep server running by default for manual API checks
STOP_SERVER_AFTER_RUN = os.environ.get('STOP_SERVER_AFTER_RUN', '0').lower() in ('1', 'true', 'yes')
if STOP_SERVER_AFTER_RUN and PID_FILE.exists():
    kill_pid(PID_FILE.read_text().strip())
    print('server stopped')
else:
    print('server kept running')
    print('base_url =', f'http://127.0.0.1:{PORT}/v1')
    print('health =', f'http://127.0.0.1:{PORT}/health')
    print('pid_file =', PID_FILE)
